# Surgical CVS AI — Colab end-to-end pipeline

Run the cells **top to bottom**. Every cell is **idempotent**: if Colab
disconnects, just re-run from the top — the repo updates in place (no
re-clone, the ~3.1 GB dataset is kept), the dataset download is cached, and
training **resumes from its last checkpoint**.

**Prerequisites**
- GPU runtime: Runtime -> Change runtime type -> GPU (T4 or better).
- Ideally the feature branch is merged into `main`; otherwise set `BRANCH`
  in the next cell to the feature-branch name.

In [ ]:
%cd /content
import os
if not os.path.isdir("sugerical-ai"):
    !git clone -b main https://github.com/duck-bin/sugerical-ai.git
%cd sugerical-ai
!git pull --ff-only
print("working dir:", os.getcwd())

In [ ]:
# Installs dependencies and removes the stale preinstalled torchao.
!bash scripts/colab_setup.sh

### (선택) Google Drive 연결 — 다운로드·체크포인트 보존

Colab은 런타임이 초기화되면 `/content`가 비워져 데이터(3GB+)와 학습 체크포인트가
사라집니다. 아래 셀은 Drive를 마운트해 `data/`·`outputs/`·모델 캐시를 Drive에
저장하므로, **다음에 다시 와도 받아둔 데이터와 학습 결과를 그대로** 씁니다.

처음 실행 시 Drive 인증 창이 뜹니다. Drive를 안 쓰려면 `USE_DRIVE = False`.

In [ ]:
USE_DRIVE = True   # Drive 안 쓰려면 False
if USE_DRIVE:
    from src.utils.colab import mount_drive
    mount_drive()

## 1. Data

CholecSeg8k (~3.1 GB) downloads from the HuggingFace Hub into the local cache;
re-running this cell is cheap once cached.

Endoscapes2023 needs PhysioNet credentialed access — download it manually to
`./data/endoscapes2023/` before the CVS step (step 4).

In [ ]:
!bash scripts/download_cholecseg8k.sh

## 2. Verify the data layer

Builds the three video-level splits. Three frame counts summing to 8080 means
the loader (and the video-id recovery) works.

In [ ]:
from src.data.cholecseg8k import CholecSeg8kDataset

for split in ("train", "val", "test"):
    print(f"{split:5s}: {len(CholecSeg8kDataset(split=split))} frames")

## 3. Train the segmentation model

`train_segmentation` defaults to **SAM2 + LoRA**; the best checkpoint is
written to `outputs/<model>/best.ckpt`.

Training is **resumable** — if the runtime disconnects, re-run this cell and it
continues from `last.ckpt`. (A full runtime *reset* wipes `/content`; to keep
checkpoints across resets, mount Google Drive and point `outputs/` at it.)

- Quick pipeline check: `!python -m src.train.train_segmentation epochs=1`
- Baseline instead of SAM2: `model=unet_baseline`

In [ ]:
!python -m src.train.train_segmentation

## 4. Train the CVS classifier

Requires the trained segmentation checkpoint from step 3 and Endoscapes2023
under `./data/endoscapes2023/`. Also resumable.

In [ ]:
!python -m src.train.train_cvs

## 5. Benchmark + demo

`benchmark_runner` evaluates the trained checkpoints and writes
`results/benchmark_table.md`. The Gradio demo is left commented out (it starts
a blocking server).

In [ ]:
!python -m src.eval.benchmark_runner
# !python -m app.gradio_demo      # interactive demo (uncomment to launch)